# HA5 Colab: SLM question answering on MIRAGE

Этот ноутбук запускается в Google Colab на GPU и считает generative SLM experiments для HA5:

- closed-book без контекста;
- oracle context из MIRAGE;
- top-1 и top-5 contexts от двух HA4 ранжировщиков (`ha4_mixture`, `ha4_dense`);
- MIRAGE mixed context.

Вход: `mirage_eval_sample_1000.jsonl` из `dz5/artifacts/data`.
Выход: `predictions/*.jsonl`, `metrics/*.json`, `qa_results_summary.csv`, zip-архив для передачи обратно в локальный `dz5`.

Сначала поставьте в Colab runtime: `Runtime -> Change runtime type -> GPU`.

In [ ]:
!nvidia-smi

## 1. Install dependencies

Если Colab попросит restart runtime после установки, перезапустите runtime и начните выполнение заново.

In [ ]:
# Keep Colab's preinstalled torch/numpy/cuda stack intact.
# Do not use -U or force-reinstall here: it can upgrade numpy/torch/cuda and break the runtime.
!pip -q install --no-deps \
  "transformers==4.51.3" \
  "accelerate==1.6.0" \
  "bitsandbytes==0.45.5" \
  "safetensors==0.5.3" \
  "tokenizers==0.21.4" \
  "huggingface-hub==0.30.2" \
  "sentencepiece==0.2.1" \
  "bert-score==0.3.13"

# Quick environment sanity check.
import numpy, torch, transformers
print('numpy', numpy.__version__)
print('torch', torch.__version__)
print('cuda available', torch.cuda.is_available())
print('transformers', transformers.__version__)


## 2. Configuration

По умолчанию стоит `Qwen/Qwen2.5-1.5B-Instruct`: модель не gated, меньше 4B и быстрее для Colab. Для более сильного, но медленного baseline можно поменять `MODEL_ID` на `Qwen/Qwen2.5-3B-Instruct`; если нужен Gemma, поменяйте `MODEL_ID` на `google/gemma-2-2b-it` и задайте `HF_TOKEN` в Colab secrets.

Для первого запуска поставьте `LIMIT = 20` или `50`. Для полного прогона поставьте `LIMIT = None`.

In [ ]:
from pathlib import Path

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"  # faster <=4B default. Higher-quality alternative: "Qwen/Qwen2.5-3B-Instruct"
RUN_PREFIX = MODEL_ID.split("/")[-1].lower().replace(".", "_").replace("-", "_")

LIMIT = None  # full fixed sample_1000; set 50 only for a deliberate smoke run
BATCH_SIZE = 8
MAX_NEW_TOKENS = 16
CONTEXT_BUDGET_CHARS = 6000
USE_4BIT = True
MIN_EXPECTED_ROWS = 1000  # protects against accidental short runs/downloads

# Checkpointing. Strongly recommended for Colab: predictions are copied to Drive after every batch.
USE_GOOGLE_DRIVE_CHECKPOINTS = True
DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/dz5_colab_checkpoints"
CHECKPOINT_EVERY_BATCHES = 1

INPUT_JSONL = Path("/content/mirage_eval_sample_1000.jsonl")
OUT_DIR = Path("/content/dz5_colab_outputs")
PRED_DIR = OUT_DIR / "predictions"
METRICS_DIR = OUT_DIR / "metrics"
TABLES_DIR = OUT_DIR / "tables"
for p in [OUT_DIR, PRED_DIR, METRICS_DIR, TABLES_DIR]:
    p.mkdir(parents=True, exist_ok=True)

EXPERIMENTS = {
    f"{RUN_PREFIX}_closed_book": {"context_mode": "none", "context_source": "none"},
    f"{RUN_PREFIX}_oracle": {"context_mode": "oracle", "context_source": "mirage"},
    f"{RUN_PREFIX}_top1_mixture": {"context_mode": "top1", "context_source": "ha4_mixture"},
    f"{RUN_PREFIX}_top1_dense": {"context_mode": "top1", "context_source": "ha4_dense"},
    f"{RUN_PREFIX}_top5_mixture": {"context_mode": "top5", "context_source": "ha4_mixture"},
    f"{RUN_PREFIX}_top5_dense": {"context_mode": "top5", "context_source": "ha4_dense"},
    f"{RUN_PREFIX}_mirage_mixed": {"context_mode": "mixed", "context_source": "mirage"},
}

# Run groups keep Colab sessions shorter. Re-run with another group; checkpoints are cumulative.
RUN_GROUP = "core_top5"  # options: core_top5, top1_optional, closed_oracle, all
RUN_GROUPS = {
    "closed_oracle": [
        f"{RUN_PREFIX}_closed_book",
        f"{RUN_PREFIX}_oracle",
    ],
    "core_top5": [
        f"{RUN_PREFIX}_closed_book",
        f"{RUN_PREFIX}_oracle",
        f"{RUN_PREFIX}_top5_mixture",
        f"{RUN_PREFIX}_top5_dense",
        f"{RUN_PREFIX}_mirage_mixed",
    ],
    "top1_optional": [
        f"{RUN_PREFIX}_top1_mixture",
        f"{RUN_PREFIX}_top1_dense",
    ],
    "all": list(EXPERIMENTS),
}
RUN_EXPERIMENT_IDS = RUN_GROUPS[RUN_GROUP]
RUN_EXPERIMENT_IDS


## 2.1. Persistent checkpoints

This mounts Google Drive and restores previous partial predictions if the Colab runtime was interrupted. Keep this enabled for full runs.


In [ ]:
import shutil, os, json

DRIVE_OUT_DIR = None
if USE_GOOGLE_DRIVE_CHECKPOINTS:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_OUT_DIR = Path(DRIVE_CHECKPOINT_DIR)
    DRIVE_OUT_DIR.mkdir(parents=True, exist_ok=True)
    print('Drive checkpoint dir:', DRIVE_OUT_DIR)

    # Restore previous outputs from Drive into /content if they exist.
    if any(DRIVE_OUT_DIR.rglob('*')):
        print('Restoring existing checkpoint files...')
        for src in DRIVE_OUT_DIR.rglob('*'):
            if src.is_file():
                rel = src.relative_to(DRIVE_OUT_DIR)
                dst = OUT_DIR / rel
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(src, dst)
        print('Restore done')
else:
    print('Drive checkpoints disabled; resume works only while the same Colab runtime is alive.')


def sync_to_drive(path: Path | None = None) -> None:
    """Copy one file or the whole OUT_DIR to Drive checkpoint storage."""
    if DRIVE_OUT_DIR is None:
        return
    if path is not None:
        if not path.exists():
            return
        rel = path.relative_to(OUT_DIR)
        dst = DRIVE_OUT_DIR / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, dst)
        return
    for src in OUT_DIR.rglob('*'):
        if src.is_file():
            rel = src.relative_to(OUT_DIR)
            dst = DRIVE_OUT_DIR / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)


## 3. Upload input data

Загрузите `mirage_eval_sample_1000.jsonl` или zip-архив `dz5_colab_input.zip`, который лежит рядом с ноутбуком в локальном `dz5/colab_bundle`.

In [ ]:
from google.colab import files
import zipfile, shutil, os

if not INPUT_JSONL.exists():
    uploaded = files.upload()
    for name in uploaded:
        src = Path("/content") / name
        if name.endswith(".zip"):
            with zipfile.ZipFile(src) as zf:
                zf.extractall("/content")
        elif name.endswith(".jsonl"):
            shutil.copy(src, INPUT_JSONL)

# Find input if it was extracted into a subfolder.
if not INPUT_JSONL.exists():
    matches = list(Path("/content").rglob("mirage_eval_sample_1000.jsonl"))
    if matches:
        shutil.copy(matches[0], INPUT_JSONL)

assert INPUT_JSONL.exists(), "Upload mirage_eval_sample_1000.jsonl or dz5_colab_input.zip"
print(INPUT_JSONL, INPUT_JSONL.stat().st_size)

## 4. Load data

In [ ]:
import json
from typing import Any

def read_jsonl(path: Path, limit: int | None = None) -> list[dict[str, Any]]:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
                if limit is not None and len(rows) >= limit:
                    break
    return rows

rows = read_jsonl(INPUT_JSONL, LIMIT)
print("LIMIT", LIMIT)
print("rows", len(rows))
if len(rows) < MIN_EXPECTED_ROWS:
    raise RuntimeError(f"Loaded only {len(rows)} rows, expected at least {MIN_EXPECTED_ROWS}. Check LIMIT and uploaded input file.")
print(rows[0].keys())
print(rows[0]["query_id"], rows[0]["question"], rows[0]["answers"][:3])

## 5. Prompt and context helpers

In [ ]:
def concat_passages(passages: list[dict[str, Any]], budget_chars: int) -> tuple[str, list[str]]:
    chunks, doc_ids = [], []
    used = 0
    for idx, passage in enumerate(passages, start=1):
        title = str(passage.get("title", "")).strip()
        text = str(passage.get("text", "")).strip()
        block = f"[{idx}] {title}\n{text}".strip()
        if not block:
            continue
        next_len = len(block) + (2 if chunks else 0)
        if chunks and used + next_len > budget_chars:
            break
        if not chunks and next_len > budget_chars:
            block = block[:budget_chars]
        chunks.append(block)
        doc_ids.append(str(passage.get("doc_id", "")))
        used += next_len
    return "\n\n".join(chunks), doc_ids


def context_for(row: dict[str, Any], config: dict[str, str], budget_chars: int) -> tuple[str, list[str]]:
    mode = config["context_mode"]
    source = config["context_source"]
    if mode == "none":
        return "", []
    if mode == "oracle":
        return str(row["oracle_context"]), [str(row["oracle_doc_id"])]
    if mode == "mixed":
        return concat_passages(row.get("mixed_contexts", []), budget_chars)
    ranker = row.get("ranker_contexts", {}).get(source, {})
    key = "top1" if mode == "top1" else "top5"
    return concat_passages(ranker.get(key, []), budget_chars)


def build_user_prompt(question: str, context: str) -> str:
    if not context.strip():
        return f"Answer the question with a short factual answer. Do not explain.\nQuestion: {question}\nAnswer:"
    return (
        "Use the context to answer the question with a short factual answer. "
        "If the context is insufficient, answer with the best short answer only.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
    )


def clean_answer(text: str) -> str:
    text = text.strip()
    for prefix in ("Answer:", "answer:"):
        if text.startswith(prefix):
            text = text[len(prefix):].strip()
    first_line = next((line.strip() for line in text.splitlines() if line.strip()), "")
    for sep in ["</s>", "<|endoftext|>", "<|im_end|>"]:
        first_line = first_line.split(sep)[0].strip()
    return first_line

## 6. Load model

Если используете gated модель (`google/gemma-2-2b-it`), добавьте `HF_TOKEN` в Colab secrets.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None

print("cuda", torch.cuda.is_available())
print("model", MODEL_ID)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

quantization_config = None
model_kwargs = {
    "device_map": "auto",
    "torch_dtype": torch.float16,
    "token": HF_TOKEN,
    "trust_remote_code": True,
}
if USE_4BIT:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )
    model_kwargs["quantization_config"] = quantization_config

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_kwargs)
model.eval()
print("loaded")

## 7. Run experiments

Файлы пишутся построчно. Если Colab оборвется, просто перезапустите эту ячейку: уже готовые `query_id` будут пропущены.

In [ ]:
from tqdm.auto import tqdm


def apply_chat_template(user_prompt: str) -> str:
    messages = [{"role": "user", "content": user_prompt}]
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        return user_prompt


def existing_qids(path: Path) -> set[str]:
    if not path.exists():
        return set()
    done = set()
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            try:
                record = json.loads(line)
                qid = record.get("query_id")
                pred = str(record.get("prediction", "")).strip()
                # Count only non-empty completed rows. Broken/truncated rows are ignored.
                if qid and pred:
                    done.add(qid)
            except Exception:
                pass
    return done


def compact_jsonl(path: Path) -> None:
    """Remove duplicate or broken rows, keeping the last completed record per query_id."""
    if not path.exists():
        return
    by_qid = {}
    order = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            try:
                record = json.loads(line)
                qid = record.get("query_id")
                pred = str(record.get("prediction", "")).strip()
                if not qid or not pred:
                    continue
                if qid not in by_qid:
                    order.append(qid)
                by_qid[qid] = record
            except Exception:
                continue
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        for qid in order:
            if qid in by_qid:
                f.write(json.dumps(by_qid[qid], ensure_ascii=False) + "\n")
    tmp.replace(path)


def batched(seq, batch_size: int):
    for i in range(0, len(seq), batch_size):
        yield seq[i:i + batch_size]


def run_experiment(experiment_id: str, config: dict[str, str]) -> Path:
    out_path = PRED_DIR / f"{experiment_id}.jsonl"
    compact_jsonl(out_path)
    sync_to_drive(out_path)

    done = existing_qids(out_path)
    todo = [row for row in rows if row["query_id"] not in done]
    print(experiment_id, "done", len(done), "todo", len(todo), "out", out_path)
    if not todo:
        return out_path

    with out_path.open("a", encoding="utf-8") as handle:
        batches = list(batched(todo, BATCH_SIZE))
        for batch_idx, batch in enumerate(tqdm(batches, desc=experiment_id), start=1):
            prompts, metadata = [], []
            for row in batch:
                context, doc_ids = context_for(row, config, CONTEXT_BUDGET_CHARS)
                prompt = apply_chat_template(build_user_prompt(row["question"], context))
                prompts.append(prompt)
                metadata.append({"row": row, "doc_ids": doc_ids})

            encoded = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to(model.device)
            input_len = encoded["input_ids"].shape[1]
            with torch.no_grad():
                output_ids = model.generate(
                    **encoded,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=False,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )
            for idx, seq in enumerate(output_ids):
                raw = tokenizer.decode(seq[input_len:], skip_special_tokens=True)
                row = metadata[idx]["row"]
                record = {
                    "experiment_id": experiment_id,
                    "query_id": row["query_id"],
                    "source": row["source"],
                    "question": row["question"],
                    "gold_answers": row["answers"],
                    "context_mode": config["context_mode"],
                    "context_source": config["context_source"],
                    "context_doc_ids": metadata[idx]["doc_ids"],
                    "prediction": clean_answer(raw),
                    "raw_output": raw,
                    "model": MODEL_ID,
                    "generation_config": {
                        "max_new_tokens": MAX_NEW_TOKENS,
                        "do_sample": False,
                        "batch_size": BATCH_SIZE,
                        "use_4bit": USE_4BIT,
                    },
                }
                handle.write(json.dumps(record, ensure_ascii=False) + "\n")
            handle.flush()
            os.fsync(handle.fileno())

            if CHECKPOINT_EVERY_BATCHES and batch_idx % CHECKPOINT_EVERY_BATCHES == 0:
                compact_jsonl(out_path)
                sync_to_drive(out_path)

    compact_jsonl(out_path)
    sync_to_drive(out_path)
    return out_path

prediction_files = []
for experiment_id in RUN_EXPERIMENT_IDS:
    prediction_files.append(run_experiment(experiment_id, EXPERIMENTS[experiment_id]))

sync_to_drive()
prediction_files

## 8. Evaluate predictions

Метрики здесь совпадают с локальным fallback evaluator: SQuAD EM/F1 и MIRAGE strict/loose. BERTScore можно включить флагом `RUN_BERTSCORE = True`, но он заметно медленнее.

In [ ]:
import collections, re, string
import numpy as np
import pandas as pd

RUN_BERTSCORE = False
ARTICLES_RE = re.compile(r"\b(a|an|the)\b", flags=re.IGNORECASE)


def normalize_squad(text: str) -> str:
    text = str(text).lower()
    text = "".join(ch for ch in text if ch not in string.punctuation)
    text = ARTICLES_RE.sub(" ", text)
    return " ".join(text.split())


def normalize_mirage(text: str) -> str:
    text = str(text).lower()
    text = "".join(ch if ch not in string.punctuation else " " for ch in text)
    return " ".join(text.split())


def squad_em(pred, gold):
    return float(normalize_squad(pred) == normalize_squad(gold))


def squad_f1(pred, gold):
    pred_tokens = normalize_squad(pred).split()
    gold_tokens = normalize_squad(gold).split()
    if not pred_tokens and not gold_tokens:
        return 1.0
    if not pred_tokens or not gold_tokens:
        return 0.0
    common = collections.Counter(pred_tokens) & collections.Counter(gold_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)


def mirage_strict(pred, gold):
    return float(normalize_mirage(pred) == normalize_mirage(gold))


def mirage_loose(pred, gold):
    pred = normalize_mirage(pred)
    ref = normalize_mirage(gold)
    if not pred or not ref:
        return float(pred == ref)
    return float(pred in ref or ref in pred)


def best(pred, golds, scorer):
    return max([scorer(pred, g) for g in golds] or [0.0])


def eval_file(path: Path) -> dict:
    preds = read_jsonl(path)
    scored = []
    for row in preds:
        pred = row.get("prediction", "")
        golds = row.get("gold_answers", [])
        scored.append({
            "query_id": row["query_id"],
            "source": row.get("source", "unknown"),
            "squad_em": best(pred, golds, squad_em),
            "squad_f1": best(pred, golds, squad_f1),
            "mirage_em_strict": best(pred, golds, mirage_strict),
            "mirage_em_loose": best(pred, golds, mirage_loose),
        })
    metrics = {k: float(np.mean([r[k] for r in scored])) if scored else 0.0 for k in ["squad_em", "squad_f1", "mirage_em_strict", "mirage_em_loose"]}
    metrics["count"] = len(scored)

    if RUN_BERTSCORE and scored:
        from bert_score import score as bert_score
        all_preds, all_refs, row_idx = [], [], []
        for i, row in enumerate(preds):
            for gold in row.get("gold_answers", []):
                all_preds.append(row.get("prediction", ""))
                all_refs.append(str(gold))
                row_idx.append(i)
        _, _, f1 = bert_score(all_preds, all_refs, lang="en", verbose=False)
        best_by_row = {}
        for i, value in zip(row_idx, f1.tolist()):
            best_by_row[i] = max(best_by_row.get(i, -1), float(value))
        metrics["bertscore_f1"] = float(np.mean(list(best_by_row.values())))

    out = METRICS_DIR / f"{path.stem}_metrics.json"
    out.write_text(json.dumps({"prediction_file": str(path), "aggregate": metrics}, indent=2, ensure_ascii=False), encoding="utf-8")
    sync_to_drive(out)
    return {"experiment_id": path.stem, **metrics}

summary = [eval_file(path) for path in sorted(PRED_DIR.glob("*.jsonl"))]
summary_df = pd.DataFrame(summary).sort_values("experiment_id")
summary_path = TABLES_DIR / "qa_results_summary.csv"
summary_df.to_csv(summary_path, index=False)
sync_to_drive(summary_path)
summary_df

## 9. Download outputs

Скачайте `dz5_colab_outputs.zip` и передайте его обратно в локальный проект. Внутри будут predictions, metrics и summary table.

## 8.1. Completion check

Run this before downloading. If any prediction file has fewer rows than expected, continue the experiment cell instead of downloading final results.


In [ ]:
def validate_completion() -> bool:
    expected = len(rows)
    ok = True
    print('Expected rows per prediction file:', expected)
    for experiment_id in RUN_EXPERIMENT_IDS:
        path = PRED_DIR / f"{experiment_id}.jsonl"
        count = 0
        unique_qids = set()
        if path.exists():
            with path.open('r', encoding='utf-8') as f:
                for line in f:
                    if not line.strip():
                        continue
                    count += 1
                    try:
                        unique_qids.add(json.loads(line).get('query_id'))
                    except Exception:
                        pass
        status = 'OK' if len(unique_qids) >= expected else 'INCOMPLETE'
        print(f'{status:10s} rows={count:5d} unique_qids={len(unique_qids):5d}/{expected:5d} {path.name}')
        if len(unique_qids) < expected:
            ok = False
    return ok

ALL_DONE = validate_completion()
if not ALL_DONE:
    raise RuntimeError('Not all experiments are complete. Re-run the experiment cell; it will resume from checkpoints. Do not download final zip yet.')
else:
    print('\nAll configured experiments are complete.')


In [ ]:
import shutil
from datetime import datetime, timezone

if 'ALL_DONE' not in globals():
    raise RuntimeError('Run the completion-check cell before downloading outputs.')
if not ALL_DONE:
    raise RuntimeError('Outputs are incomplete. Re-run experiment cell before downloading.')

def count_completed_qids(path: Path) -> dict[str, int]:
    rows_count = 0
    unique_qids = set()
    if path.exists():
        with path.open('r', encoding='utf-8') as f:
            for line in f:
                if not line.strip():
                    continue
                rows_count += 1
                try:
                    qid = json.loads(line).get('query_id')
                    if qid:
                        unique_qids.add(qid)
                except Exception:
                    pass
    return {'rows': rows_count, 'unique_qids': len(unique_qids)}

manifest = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'model_id': MODEL_ID,
    'run_prefix': RUN_PREFIX,
    'limit': LIMIT,
    'min_expected_rows': MIN_EXPECTED_ROWS,
    'rows_loaded': len(rows),
    'expected_rows_per_experiment': len(rows),
    'batch_size': BATCH_SIZE,
    'max_new_tokens': MAX_NEW_TOKENS,
    'use_4bit': USE_4BIT,
    'use_google_drive_checkpoints': USE_GOOGLE_DRIVE_CHECKPOINTS,
    'drive_checkpoint_dir': DRIVE_CHECKPOINT_DIR,
    'run_group': RUN_GROUP,
    'run_experiment_ids': RUN_EXPERIMENT_IDS,
    'all_done': ALL_DONE,
    'prediction_counts': {
        experiment_id: count_completed_qids(PRED_DIR / f'{experiment_id}.jsonl')
        for experiment_id in RUN_EXPERIMENT_IDS
    },
}
manifest_path = OUT_DIR / 'run_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')
sync_to_drive(manifest_path)

sync_to_drive()
archive_base = "/content/dz5_colab_outputs"
zip_path = shutil.make_archive(archive_base, "zip", OUT_DIR)
print(zip_path)
files.download(zip_path)
